# Can we predict spam?
The data looks like the UCI Spambase set: 57 features (word/character frequencies, capital-letter run lengths) and `Class` (1 = spam). The notebook checks the data, sets a baseline, compares models with cross-validation, then evaluates the chosen one once on a held-out test set.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (classification_report, ConfusionMatrixDisplay, RocCurveDisplay,
                             roc_auc_score, average_precision_score)
from sklearn.inspection import permutation_importance
RS = 42
df = pd.read_csv('data.csv')
print(df.shape); df.head()

## 1. Data checks

In [ ]:
print('missing values:', int(df.isna().sum().sum()))
print('non-numeric columns:', list(df.select_dtypes(exclude='number').columns))
print('duplicate rows:', int(df.duplicated().sum()))
print(df['Class'].value_counts().rename({0: 'ham', 1: 'spam'}))
print(df['Class'].value_counts(normalize=True).round(3).to_dict())

Duplicate rows would leak between train and test (the same email on both sides) and inflate scores, so I drop them before splitting.

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
X, y = df.drop(columns='Class'), df['Class']
print(df.shape, 'spam share: %.3f' % y.mean())

## 2. Split
Stratified 80/20. The test set is touched only once, at the end.

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RS)
print(X_tr.shape, X_te.shape)

## 3. A quick look at the signal

In [ ]:
d = X_tr.assign(Class=y_tr).groupby('Class').mean().T
d['ratio'] = (d[1] + 1e-3) / (d[0] + 1e-3)
top = pd.concat([d.sort_values('ratio').head(8), d.sort_values('ratio').tail(8)])
ax = np.log2(top['ratio']).plot.barh(figsize=(7, 6), color=['#2a6f97' if r < 1 else '#c0392b' for r in top['ratio']])
ax.set_xlabel('log2(mean in spam / mean in ham)'); ax.set_title('Most spam- and ham-leaning features'); plt.tight_layout()

## 4. Models (5-fold CV on the training set)
Frequencies are heavily right-skewed, so the linear model gets a `log1p` transform. Trees don't need it.

In [ ]:
log = FunctionTransformer(np.log1p)
models = {
    'baseline (majority)': DummyClassifier(strategy='most_frequent'),
    'logistic regression': make_pipeline(log, StandardScaler(), LogisticRegression(max_iter=2000, C=1.0)),
    'random forest': RandomForestClassifier(n_estimators=400, n_jobs=-1, random_state=RS),
    'gradient boosting': HistGradientBoostingClassifier(random_state=RS),
}
cv = StratifiedKFold(5, shuffle=True, random_state=RS)
rows = {}
for n, m in models.items():
    r = cross_validate(m, X_tr, y_tr, cv=cv, scoring=['accuracy', 'precision', 'recall', 'f1', 'roc_auc'])
    rows[n] = {k[5:]: v.mean() for k, v in r.items() if k.startswith('test_')}
cvres = pd.DataFrame(rows).T.round(4)
cvres

## 5. Final evaluation on the held-out test set

In [ ]:
best = 'gradient boosting' if cvres.loc['gradient boosting', 'roc_auc'] >= cvres.loc['random forest', 'roc_auc'] else 'random forest'
print('chosen by CV ROC-AUC:', best)
model = models[best].fit(X_tr, y_tr)
proba = model.predict_proba(X_te)[:, 1]; pred = (proba >= .5).astype(int)
print(classification_report(y_te, pred, target_names=['ham', 'spam'], digits=3))
print('ROC-AUC %.4f | PR-AUC %.4f' % (roc_auc_score(y_te, proba), average_precision_score(y_te, proba)))
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ConfusionMatrixDisplay.from_predictions(y_te, pred, display_labels=['ham', 'spam'], ax=ax[0], colorbar=False)
RocCurveDisplay.from_predictions(y_te, proba, ax=ax[1]); ax[1].plot([0, 1], [0, 1], 'k:')
plt.tight_layout()

### Threshold trade-off
Wrongly flagging a real email (false positive) usually costs more than letting spam through, and the default 0.5 cut-off isn't sacred. This table shows what moving it does. (Computed on the test set for illustration only; in practice choose the threshold with CV on training data.)

In [ ]:
rows = []
for t in [.3, .4, .5, .6, .7, .8, .9]:
    p = (proba >= t).astype(int)
    rows.append(dict(threshold=t, false_pos=int(((p == 1) & (y_te == 0)).sum()), missed_spam=int(((p == 0) & (y_te == 1)).sum()),
                     precision=y_te[p == 1].mean(), recall=p[y_te == 1].mean()))
pd.DataFrame(rows).round(3)

## 6. What drives the predictions?

In [ ]:
pi = permutation_importance(model, X_te, y_te, scoring='roc_auc', n_repeats=10, random_state=RS, n_jobs=-1)
imp = pd.Series(pi.importances_mean, index=X.columns).sort_values().tail(15)
imp.plot.barh(figsize=(7, 5), xerr=pd.Series(pi.importances_std, index=X.columns)[imp.index])
plt.xlabel('drop in ROC-AUC when shuffled'); plt.title('Permutation importance (test set)'); plt.tight_layout()

## Summary

In [ ]:
print(f'Majority baseline accuracy (CV): {cvres.loc["baseline (majority)", "accuracy"]:.3f}')
print(f'{best}: test accuracy {(pred == y_te).mean():.3f}, ROC-AUC {roc_auc_score(y_te, proba):.3f}')